In [52]:
import torch
import torch.nn as nn
import pandas as pd
import math
import random

In [53]:
base_vocab = torch.load("base_vocab.pt")
base_words = torch.load("base_words.pt")

transformer_vocab = torch.load("transformer_vocab.pt")
transformer_words = torch.load("transformer_words.pt")

In [54]:
PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
UNK = "<UNK>"

In [55]:
device = torch.device("cuda"if torch.cuda.is_available()else "cpu")

In [56]:
BASE_VOCAB_SIZE = len(base_vocab)
TRANSFORMER_VOCAB_SIZE = len(transformer_vocab)

EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1
BASE_PAD_IDX = base_vocab[PAD]
TRANSFORMER_PAD_IDX = transformer_vocab[PAD]
NHEAD = 8
FF_DIM = 512

In [57]:
def tokenizer(code):
    return code.split()

In [58]:
def encode_base(tokens):
    return torch.tensor([base_vocab.get(token,base_vocab[UNK])for token in tokens],dtype=torch.long)

In [59]:
def encode_transformer(tokens):
    return torch.tensor([transformer_vocab.get(token,transformer_vocab[UNK])for token in tokens],dtype=torch.long)

In [60]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 3,hidden_dim)
        self.v = nn.Linear(hidden_dim,1,bias=False)
   
    def forward(self,hidden,encoder_outputs):
        src_len = encoder_outputs.shape[0]
        hidden = hidden.repeat(src_len,1,1)
        energy = torch.tanh(self.attn(torch.cat((hidden,encoder_outputs),dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention,dim=0)

In [61]:
class AttentionBiEncoder(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim,padding_idx=BASE_PAD_IDX)
        self.lstm = nn.LSTM(embedding_dim,hidden_dim,num_layers,bidirectional=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        hidden = (hidden[0:hidden.size(0):2]+hidden[1:hidden.size(0):2])
        cell = (cell[0:cell.size(0):2]+cell[1:cell.size(0):2])

        return outputs, hidden, cell

In [62]:
class AttentionDecoder(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,attention):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim,padding_idx=BASE_PAD_IDX)
        self.attention = attention
        self.lstm = nn.LSTM(embedding_dim + hidden_dim * 2,hidden_dim)
        self.fc = nn.Linear(hidden_dim,vocab_size)
    
    def forward(self,input_token,hidden,cell,encoder_outputs):
        input_token = input_token.unsqueeze(0)
        embedded = self.embedding(input_token)
        attn_weights = self.attention(hidden[-1].unsqueeze(0),encoder_outputs)
        context = torch.sum(attn_weights.unsqueeze(2)*encoder_outputs,dim=0)
        context = context.unsqueeze(0)
        lstm_input = torch.cat((embedded,context),dim=2)
        output, (hidden, cell) = self.lstm(lstm_input,(hidden, cell))
        prediction = self.fc(output.squeeze(0))

        return prediction, hidden, cell

In [63]:
class AttentionSeq2Seq(nn.Module):

    def __init__(self,encoder,decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self,src,trg,teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(trg_len,batch_size,vocab_size,device=src.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        input_token = trg[0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_token,hidden,cell,encoder_outputs)
            outputs[t] = output
            best_guess = output.argmax(1)
            teacher_force = (random.random()<teacher_forcing_ratio)
            input_token = ( trg[t] if teacher_force else best_guess )

        return outputs

In [64]:
class Positional(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0,max_len,dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)
        self.register_buffer("pe",pe)
    def forward(self, x):
        return (x+self.pe[:x.size(0)])

In [65]:
class SelfAttention(nn.Module):

    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, key, value, mask=None):
        Q = self.Wq(query)
        K = self.Wk(key)
        V = self.Wv(value)
        Q = Q.permute(1,0,2)
        K = K.permute(1,0,2)
        V = V.permute(1,0,2)
        scores = torch.matmul(Q,K.transpose(-2, -1)) / math.sqrt(self.embed_dim)
        if mask is not None:scores = scores.masked_fill(mask == 0,float("-inf"))
        attention = torch.softmax(scores,dim=-1)
        output = torch.matmul(attention,V)
        output = output.permute(1,0,2)
        return output

In [66]:
class HeadAttention(nn.Module):
    def __init__(self, embed_dim, nhead):
        super().__init__()
        self.head_dim = embed_dim//nhead
        self.nhead = nhead
        self.heads = nn.ModuleList([SelfAttention(self.head_dim) for _ in range(nhead)])
        self.fc = nn.Linear(embed_dim,embed_dim)
        
    def forward(self,query,key,value,mask=None):
        batch_size = query.shape[1]
        outputs = []
        for i, head in enumerate(self.heads):
            start = i * self.head_dim
            end = start + self.head_dim
            q = query[:, :, start:end]
            k = key[:, :, start:end]
            v = value[:, :, start:end]
            outputs.append(head(q,k,v,mask))
        output = torch.cat(outputs,dim=-1)
        return self.fc(output)

In [67]:
class Forward(nn.Module):
    def __init__(self,embed_dim,ff_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

    def forward(self, x):
        return self.net(x)

In [68]:
class Encoder(nn.Module):
    def __init__(self,embed_dim,nhead,ff_dim,dropout):
        super().__init__()
        self.attn = HeadAttention(embed_dim,nhead)
        self.ff = Forward(embed_dim,ff_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        attn = self.attn(src,src,src)
        src = self.norm1(src +self.dropout(attn))
        ff = self.ff(src)
        src = self.norm2(src +self.dropout(ff))
        return src

In [69]:
class Decoder(nn.Module):
    def __init__(self,embed_dim,nhead,ff_dim,dropout):
        super().__init__()
        self.self_attn = HeadAttention(embed_dim,nhead)
        self.cross_attn = HeadAttention(embed_dim,nhead)
        self.ff = Forward(embed_dim,ff_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.norm3 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self,trg,memory,mask):
        attn = self.self_attn(trg,trg,trg,mask)
        trg = self.norm1(trg +self.dropout(attn))
        attn = self.cross_attn(trg,memory,memory)
        trg = self.norm2(trg +self.dropout(attn))
        ff = self.ff(trg)
        trg = self.norm3(trg +self.dropout(ff))
        return trg

In [70]:
class Transformer(nn.Module):
    def __init__(self,vocab_size,embed_dim,nhead,num_layers,ff_dim,dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_dim,padding_idx=TRANSFORMER_PAD_IDX)
        self.pos_encoder = Positional(embed_dim)
        self.encoder_layers = nn.ModuleList([Encoder(embed_dim,nhead,ff_dim,dropout)for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([Decoder(embed_dim,nhead,ff_dim,dropout)for _ in range(num_layers)])
        self.fc = nn.Linear(embed_dim,vocab_size)
   
    def make_mask(self,trg_len,device):
        mask = torch.tril(torch.ones(trg_len,trg_len,device=device))
        return mask

    def forward(self,src,trg):
        src = self.embedding(src)
        trg = self.embedding(trg)
        src = self.pos_encoder(src)
        trg = self.pos_encoder(trg)
        memory = src
        for layer in self.encoder_layers:
            memory = layer(memory)
        mask = self.make_mask(trg.size(0),trg.device)
        output = trg
        for layer in self.decoder_layers:
            output = layer(output,memory,mask)
        return self.fc(output)

In [71]:
def repair_attention(code,model,max_len=100):
    model.eval()
    tokens = tokenizer(code)
    src = encode_base(tokens).unsqueeze(1).to(device)
    with torch.no_grad():
        encoder_outputs, hidden, cell = (model.encoder(src))
        input_token = torch.tensor([base_vocab[SOS]],dtype=torch.long,device=device)
        generated = []
        for _ in range(max_len):
            prediction, hidden, cell = (model.decoder(input_token,hidden,cell,encoder_outputs))
            best_guess = prediction.argmax(1)
            token_id = best_guess.item()
            if token_id == base_vocab[EOS]:
                break
            generated.append(base_words[token_id])
            input_token = best_guess

    return " ".join(generated)

In [72]:
def repair_transformer(code, model, max_len=100):
    model.eval()
    tokens = tokenizer(code)
    src = encode_transformer(tokens).unsqueeze(1).to(device)
    generated = [transformer_vocab[SOS]]
    with torch.no_grad():
        for _ in range(max_len):
            trg = torch.tensor(generated,dtype=torch.long,device=device).unsqueeze(1)
            output = model(src, trg)
            next_token = output[-1].argmax(1).item()
            if next_token == transformer_vocab[EOS]:
                break
            generated.append(next_token)
    return " ".join(transformer_words[idx]for idx in generated[1:])

In [73]:
attention = Attention(HIDDEN_DIM)

attention_model = AttentionSeq2Seq(
    AttentionBiEncoder(BASE_VOCAB_SIZE,EMBED_DIM,HIDDEN_DIM,NUM_LAYERS),
    AttentionDecoder(BASE_VOCAB_SIZE,EMBED_DIM,HIDDEN_DIM,attention)
).to(device)

attention_model.load_state_dict(torch.load("bilstm_attention.pth",map_location=device))

attention_model.eval()

AttentionSeq2Seq(
  (encoder): AttentionBiEncoder(
    (embedding): Embedding(415, 128, padding_idx=0)
    (lstm): LSTM(128, 256, bidirectional=True)
  )
  (decoder): AttentionDecoder(
    (embedding): Embedding(415, 128, padding_idx=0)
    (attention): Attention(
      (attn): Linear(in_features=768, out_features=256, bias=True)
      (v): Linear(in_features=256, out_features=1, bias=False)
    )
    (lstm): LSTM(640, 256)
    (fc): Linear(in_features=256, out_features=415, bias=True)
  )
)

In [76]:
NUM_LAYERS=3
transformer_model = Transformer(TRANSFORMER_VOCAB_SIZE,EMBED_DIM,NHEAD,NUM_LAYERS,FF_DIM).to(device)
transformer_model.load_state_dict(torch.load("transformer.pth",map_location=device))

transformer_model.eval()

Transformer(
  (embedding): Embedding(415, 128, padding_idx=0)
  (pos_encoder): Positional()
  (encoder_layers): ModuleList(
    (0-2): 3 x Encoder(
      (attn): HeadAttention(
        (heads): ModuleList(
          (0-7): 8 x SelfAttention(
            (Wq): Linear(in_features=16, out_features=16, bias=True)
            (Wk): Linear(in_features=16, out_features=16, bias=True)
            (Wv): Linear(in_features=16, out_features=16, bias=True)
          )
        )
        (fc): Linear(in_features=128, out_features=128, bias=True)
      )
      (ff): Forward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): ReLU()
          (2): Linear(in_features=512, out_features=128, bias=True)
        )
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.3, inplace=False)
    )
  )
  (decoder_layers): ModuleList(
  

In [80]:
buggy = input("Enter buggy code:\n")
print("\nBUGGY:")
print(buggy)
print("\nBiLSTM + Attention:")
print(repair_attention(buggy,attention_model))
print("\nTransformer:")
print(repair_transformer(buggy,transformer_model))


BUGGY:
if ( VAR_1 == null ) return ;

BiLSTM + Attention:
( ) VAR_1 ) ; return ; }

Transformer:
: ( ) { return ( ) == null ; if ( ( ) == null ) return VAR_1 == null ; if ( ( ) == null ) return VAR_1 == null : return VAR_1 ; }
